# Training on a TPU in colab

Trains the transformer on a Colab TPU runtime (such as `TPU v5e-1` or `TPU v6e-1`) using `torch_tpu`. See `colab_train.ipynb` for GPUs; dataset preparation is identical.

### Runtime & TPU v5e-1 notes
Select a TPU runtime with high RAM (e.g. `v5e-1` or `v6e-1`). `training/transformer/train.py` detects `torch_tpu` (`torch.device("tpu")`) and configures `transformers` 4.x / `accelerate` 1.x to use it:

- compiles the model with `torch.compile(..., backend="tpu")` (lowered to StableHLO / OpenXLA),
- uses `adamw_torch` instead of CUDA-only fused AdamW and runs the forward pass under `torch.autocast("tpu", dtype=torch.bfloat16)`,
- drops the last incomplete batch (`dataloader_drop_last=True`) so a smaller tail batch does not trigger extra XLA graph compilations.

The first steps are slow while `torch.compile` compiles one graph per sequence-length bucket (multiples of 64 tokens up to 608) for training, the scheduled-sampling pass, and evaluation, plus once more when the ConvNeXt backbone is unfrozen at epoch 2.

### Batch size
`TPU v5e-1` has 16 GB HBM (`v6e-1` has 32 GB), matching the default `batch_size = 8` / `16` setup for 16 GB VRAM. If you increase `batch_size`, adjust `gradient_accumulation_steps` accordingly.

In [ ]:
# MOUNT GDRIVE FOR DATASET DOWNLOAD
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# INSTALL HOMR
!pip install -q uv

# if you're training on your fork replace this with it
!git clone https://github.com/liebharc/homr.git
%cd /content/homr

# switch to your branch
!git switch feature/my_branch
!mkdir -p datasets

# onnx-simplifier is rebuild every time which makes this step take
# really long, you should therefore remove it from pyproject.toml
!uv pip install --system -r pyproject.toml --group dev --extra tpu --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
# COPY DATASETS TO DISK
import shutil

source = '/content/drive/MyDrive/datasets' # may be a different location for you
destination = '/content/homr/datasets/'

shutil.copytree(source, destination, dirs_exist_ok=True)

In [ ]:
# EXTRACT DATASETS
!7z x /content/homr/datasets/lieder.7z -o/content/homr/datasets/Lieder-main
!7z x /content/homr/datasets/pdmx.7z -o/content/homr/datasets/pdmx
!7z x /content/homr/datasets/grandstaff_data.7z -o/content/homr/datasets/grandstaff

In [ ]:
# CHECKS
!python3 -c "import torch, torch_tpu; print(torch.__version__, torch.accelerator.current_accelerator(), 'available:', torch.accelerator.is_available(), 'devices:', torch.accelerator.device_count())"
!nproc # number threads for dataloader_num_workers

In [ ]:
# TRAIN TRANSFORMER
# Make sure you set batch_size, gradient_accumulation_steps and dataloader_num_workers correctly.
# Data augmentation runs on the CPU, so set dataloader_num_workers > 0 in train.py.
# Also create frequent backups of the current_training folder so you don't lose progress
# when you get disconnected.
!python3 -m training.train transformer